## Mode Selection

In [1]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection'}


## Setup & Model Loading

In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


Found Triton wheels: ['/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec： ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7b051dd48260>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


Training environment fixes applied.


In [4]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.


In [5]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


Found mamba wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Found causal-conv1d wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Selected causal wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Selected mamba wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-25 17:59:18.353702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777139958.567900      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777139958.629131      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777139959.151673      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777139959.151688      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777139959.151689      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # ============================================================
    # OPTIMIZED LoRA CONFIG (v8) — targeting 92%+ accuracy
    # ============================================================
    # Rank 32 is the competition maximum.
    # RSLoRA uses alpha/sqrt(r) scaling — already regularizes,
    # so dropout=0.0 avoids double-regularization penalty.
    # lm_head REMOVED: prevents format drift on \\boxed{} output.
    # ============================================================
    LORA_RANK = 32
    LORA_ALPHA = 32          # 2x rank — strong adapter influence
    LORA_DROPOUT = 0.0       # RSLoRA already regularizes; dropout hurts

    target_modules = [
        # Attention projections (core reasoning)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj",

        # MLP / MoE (feed-forward reasoning capacity)
        "gate_proj", "up_proj", "down_proj",

        # Mamba-specific (critical for Nemotron's hybrid architecture)
        "x_proj", "dt_proj",
        "lm_head"   
        # NOTE: lm_head intentionally EXCLUDED to prevent
        # output distribution drift on \\boxed{} formatting
    ]

    print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
    print(f"  Rank={LORA_RANK}, Alpha={LORA_ALPHA}, Dropout={LORA_DROPOUT}")
    print(f"  Target modules: {target_modules}")
    print(f"  RSLoRA=True, lm_head=EXCLUDED")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")


Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...
  Rank=32, Alpha=64, Dropout=0.0
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'gate_proj', 'up_proj', 'down_proj', 'x_proj', 'dt_proj']
  RSLoRA=True, lm_head=EXCLUDED
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'gate_proj', 'up_proj', 'down_proj', 'x_proj', 'dt_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## Mode A: Train on Kaggle

In [8]:
if TRAIN_ON_KAGGLE:
    # ============================================================
    # MEMORY OPTIMIZATIONS
    # FIX: Removed TORCH_CUDA_ALLOC_CONF which conflicted and
    #      caused memory fragmentation → CUDA illegal memory access
    # ============================================================
    import os
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

    import pandas as pd
    import random
    import gc
    import time
    import torch
    import re
    import math
    import subprocess
    import zipfile
    from pathlib import Path
    from collections import defaultdict
    from torch.utils.data import DataLoader, Sampler
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback

    # ============================================================
    # GPU METRICS CALLBACK (TensorBoard)
    # Logs: GPU util%, memory, temperature, power, throughput
    # ============================================================
    class GPUMetricsCallback(TrainerCallback):
        def __init__(self, log_every_n_steps=2):
            super().__init__()
            self.log_every_n_steps = log_every_n_steps
            self._last_step_time = None
            self._last_global_step = 0

        def _query_nvidia_smi(self):
            try:
                r = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=utilization.gpu,memory.used,memory.total,"
                     "temperature.gpu,power.draw",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5)
                if r.returncode != 0: return None
                p = [x.strip() for x in r.stdout.strip().split("\n")[0].split(",")]
                return {
                    "gpu/utilization_percent": float(p[0]),
                    "gpu/memory_used_gb": float(p[1]) / 1024.0,
                    "gpu/temperature_celsius": float(p[3]),
                    "gpu/power_watts": float(p[4]) if p[4] != "[N/A]" else 0.0,
                }
            except Exception:
                return None

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is None or state.global_step % self.log_every_n_steps != 0:
                return
            smi = self._query_nvidia_smi()
            if smi:
                logs.update(smi)
            if torch.cuda.is_available():
                logs["gpu/memory_allocated_gb"] = torch.cuda.memory_allocated() / (1024**3)
                logs["gpu/memory_reserved_gb"] = torch.cuda.memory_reserved() / (1024**3)
                logs["gpu/max_memory_allocated_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
            now = time.time()
            if self._last_step_time is not None:
                elapsed = now - self._last_step_time
                steps = state.global_step - self._last_global_step
                if elapsed > 0 and steps > 0:
                    sps = steps / elapsed
                    logs["throughput/steps_per_sec"] = sps
                    logs["throughput/samples_per_sec"] = sps * args.per_device_train_batch_size
            self._last_step_time = now
            self._last_global_step = state.global_step

        def on_train_begin(self, args, state, control, **kwargs):
            self._last_step_time = time.time()
            self._last_global_step = state.global_step
            print("[TensorBoard] GPU metrics logging enabled.")
            smi = self._query_nvidia_smi()
            if smi:
                print(f"  GPU: {smi['gpu/utilization_percent']:.0f}% util | "
                      f"{smi['gpu/memory_used_gb']:.1f} GB | "
                      f"{smi['gpu/temperature_celsius']:.0f}C")

        def on_train_end(self, args, state, control, **kwargs):
            smi = self._query_nvidia_smi()
            if smi:
                peak = torch.cuda.max_memory_allocated() / (1024**3)
                print(f"[TensorBoard] Done. Peak mem: {peak:.1f} GB | "
                      f"Temp: {smi['gpu/temperature_celsius']:.0f}C")

    # ============================================================
    # DATA LOADING
    # ============================================================
    SEED = 42
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")

    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Shuffled dataset: {len(train_df)} rows")

    records = []
    record_types = []
    skipped = 0
    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])
        if not cot or cot == "nan" or len(cot.strip()) < 5 or len(cot.strip()) > 8100:
            skipped += 1
            continue
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        user_content = prompt + PROMPT_SUFFIX
        assistant_content = "<think>\n" + cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
        records.append({
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content},
            ]
        })
        record_types.append(str(row["type"]))

    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)} (skipped {skipped} invalid CoT)")

    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages
        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation, tokenize=False,
                    add_generation_prompt=False, enable_thinking=True)
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return texts

    # ============================================================
    # TRAINING CONFIG v8.1 (crash-fixed + TensorBoard)
    # Fixes: adamw_8bit (was paged), removed TORCH_CUDA_ALLOC_CONF
    # ============================================================
    TB_LOG_DIR = "/kaggle/working/tb_logs"

    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_length=8192,
        optim="adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        weight_decay=0.005,
        max_grad_norm=0.5,
        neftune_noise_alpha=5.0,
        logging_steps=2,
        logging_dir=TB_LOG_DIR,
        report_to="tensorboard",
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        packing=False,
        dataset_num_proc=4,
    )

    print("\n" + "="*60)
    print("  TRAINING CONFIG v8.1 (crash-fixed + TensorBoard)")
    print("="*60)
    print(f"  LR:           {training_args.learning_rate}")
    print(f"  Epochs:       {training_args.num_train_epochs}")
    print(f"  Warmup:       {training_args.warmup_ratio}")
    print(f"  NEFTune:      {training_args.neftune_noise_alpha}")
    bs = training_args.per_device_train_batch_size
    ga = training_args.gradient_accumulation_steps
    print(f"  Batch:        {bs} x {ga} = {bs * ga}")
    print(f"  Optimizer:    {training_args.optim} (crash fix)")
    print(f"  TensorBoard:  {TB_LOG_DIR}")
    print("="*60 + "\n")

    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)
        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)
        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)
        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1
        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)
        def __iter__(self):
            return iter(self.order)
        def __len__(self):
            return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order
        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            if len(self.stratified_order) != len(self.train_dataset):
                raise ValueError("Stratified order length mismatch")
            kw = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                kw["prefetch_factor"] = self.args.dataloader_prefetch_factor
            return DataLoader(self.train_dataset, **kw)

    effective_batch_size = max(1, bs * ga)
    stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
    print(f"Stratified effective batch size: {effective_batch_size}")
    print("Types:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

    trainer = StratifiedSFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
        callbacks=[GPUMetricsCallback(log_every_n_steps=2)],
    )

    torch.cuda.empty_cache()
    gc.collect()

    print("Starting SFT training v8.1 (crash-fixed + TensorBoard)...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed/60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")


Full dataset: 7830 rows
Shuffled dataset: 7830 rows
SFT records: 5227 (skipped 2603 invalid CoT)

  TRAINING CONFIG v8.1 (crash-fixed + TensorBoard)
  LR:           0.0002
  Epochs:       2
  Warmup:       0.1
  NEFTune:      5.0
  Batch:        2 x 4 = 8
  Optimizer:    OptimizerNames.ADAMW_8BIT (crash fix)
  TensorBoard:  /kaggle/working/tb_logs

Stratified effective batch size: 8
Types: {'bit_manipulation': 91, 'cipher': 1500, 'cryptarithm_deduce': 627, 'cryptarithm_guess': 154, 'gravity': 1055, 'numeral': 730, 'unit_conversion': 1070}


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/5227 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting SFT training v8.1 (crash-fixed + TensorBoard)...
[TensorBoard] GPU metrics logging enabled.
  GPU: 0% util | 62.9 GB | 40C


Step,Training Loss
2,2.409000
4,2.421300
6,2.126500
8,1.540500
10,1.151500
12,1.033500
14,1.194800
16,1.237200
18,0.920500
20,0.808600


[TensorBoard] Done. Peak mem: 86.6 GB | Temp: 48C
Training done in 393.3 min
Adapter saved to /kaggle/working/sft_adapter


## Package TensorBoard Logs for Download

In [9]:
if TRAIN_ON_KAGGLE:
    import os, zipfile
    from pathlib import Path

    TB_LOG_DIR = "/kaggle/working/tb_logs"
    ZIP_OUTPUT = "/kaggle/working/tensorboard_logs.zip"

    log_path = Path(TB_LOG_DIR)
    if log_path.exists():
        files = [f for f in log_path.rglob("*") if f.is_file()]
        events = list(log_path.rglob("events.out.tfevents.*"))
        print(f"\n{'='*60}")
        print(f"  PACKAGING TENSORBOARD LOGS ({len(events)} event files, {len(files)} total)")
        print(f"{'='*60}")
        with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in files:
                zf.write(fp, fp.relative_to(log_path.parent))
        sz = os.path.getsize(ZIP_OUTPUT) / (1024*1024)
        print(f"  => {ZIP_OUTPUT} ({sz:.2f} MB)")
        print(f"\n  TO VIEW LOCALLY:")
        print(f"  unzip tensorboard_logs.zip")
        print(f"  pip install tensorboard")
        print(f"  tensorboard --logdir=tb_logs/ --port=6006")
        print(f"  # Open http://localhost:6006")
        print(f"{'='*60}\n")
    else:
        print(f"[WARN] No TB logs at {TB_LOG_DIR}")



  PACKAGING TENSORBOARD LOGS (1 event files, 1 total)
  => /kaggle/working/tensorboard_logs.zip (0.04 MB)

  TO VIEW LOCALLY:
  unzip tensorboard_logs.zip
  pip install tensorboard
  tensorboard --logdir=tb_logs/ --port=6006
  # Open http://localhost:6006



## Mode B: Load Pre-trained LoRA（Temporarily unavailable）

In [10]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.


## Create submission.zip

In [11]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")


Packaging freshly trained adapter from: /kaggle/working/sft_adapter
Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (3373.4 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3098.8 MB
Done! Ready to submit.
